In [4]:
# ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
# A suitable version of pyarrow or fastparquet is required for parquet support.
# Trying to import the above resulted in these errors:
#  - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
#  - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.
# Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings... 
!pip install pyarrow
!pip install fastparquet

   ---------------------------------------- 0.0/27.5 MB ? eta -:--:--
   --- ------------------------------------ 2.6/27.5 MB 16.7 MB/s eta 0:00:02
   --------- ------------------------------ 6.3/27.5 MB 16.8 MB/s eta 0:00:02
   ------------- -------------------------- 9.2/27.5 MB 15.4 MB/s eta 0:00:02
   ----------------- ---------------------- 12.1/27.5 MB 15.1 MB/s eta 0:00:02
   --------------------- ------------------ 14.9/27.5 MB 14.7 MB/s eta 0:00:01
   ------------------------- -------------- 17.8/27.5 MB 14.6 MB/s eta 0:00:01
   ----------------------------- ---------- 20.4/27.5 MB 14.2 MB/s eta 0:00:01
   --------------------------------- ------ 23.3/27.5 MB 14.1 MB/s eta 0:00:01
   ------------------------------------- -- 26.0/27.5 MB 13.8 MB/s eta 0:00:01
   ---------------------------------------  27.3/27.5 MB 13.8 MB/s eta 0:00:01
   ---------------------------------------- 27.5/27.5 MB 13.2 MB/s  0:00:02



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/669.8 kB ? eta -:--:--
   ---------------------------------------- 669.8/669.8 kB 13.1 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 13.2 MB/s  0:00:00

   -------------------- ------------------- 1/2 [fastparquet]
   -------------------- ------------------- 1/2 [fastparquet]
   ---------------------------------------- 2/2 [fastparquet]




[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import io
import requests
import pandas as pd

def load_data(*args, **kwargs):
    year = kwargs.get('year', 2025)
    month = kwargs.get('month', '01')

    url = f'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month}.parquet'

    # Some institutional networks block generic urllib traffic and return HTTP 403.
    headers = {
        'User-Agent': kwargs.get(
            'user_agent',
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
            '(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36'
        )
    }

    response = requests.get(url, headers=headers, timeout=60)
    response.raise_for_status()

    parquet_buffer = io.BytesIO(response.content)

    engine = kwargs.get('engine', 'fastparquet')
    try:
        datos_crudos = pd.read_parquet(parquet_buffer, engine=engine)
    except Exception:
        parquet_buffer.seek(0)
        datos_crudos = pd.read_parquet(parquet_buffer, engine='fastparquet')

    return datos_crudos

data = load_data(year=2025, month='01')
data.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


In [2]:
data.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee'],
      dtype='object')

In [3]:
def transform(data, *args, **kwargs):
    data.columns = [col.lower() for col in data.columns]
    
    data.rename(columns={
        'vendorid': 'vendor_id',
        'ratecodeid': 'rate_code_id',
        'pulocationid': 'pu_location_id',
        'dolocationid': 'do_location_id'
    }, inplace = True)

    return data

data = transform(data)
data.columns

Index(['vendor_id', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'rate_code_id',
       'store_and_fwd_flag', 'pu_location_id', 'do_location_id',
       'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount',
       'tolls_amount', 'improvement_surcharge', 'total_amount',
       'congestion_surcharge', 'airport_fee', 'cbd_congestion_fee'],
      dtype='object')

In [4]:
data.head()

,vendor_id,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pu_location_id,do_location_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


In [20]:
rows = data.shape[0]
chunksize = 500000

print(f'Total rows: {rows}')
print(f'Chunk size: {chunksize}')

print("Replace")

print(f'Chunk {0//chunksize + 1}: {chunksize} rows')

print("Append")

for i in range(chunksize, rows, chunksize):
    chunk = data.iloc[i:i+chunksize]
    print(f'Chunk {i//chunksize + 1}: {chunk.shape[0]} rows')

Total rows: 12741035
Chunk size: 500000
Replace
Chunk 1: 500000 rows
Append
Chunk 2: 500000 rows
Chunk 3: 500000 rows
Chunk 4: 500000 rows
Chunk 5: 500000 rows
Chunk 6: 500000 rows
Chunk 7: 500000 rows
Chunk 8: 500000 rows
Chunk 9: 500000 rows
Chunk 10: 500000 rows
Chunk 11: 500000 rows
Chunk 12: 500000 rows
Chunk 13: 500000 rows
Chunk 14: 500000 rows
Chunk 15: 500000 rows
Chunk 16: 500000 rows
Chunk 17: 500000 rows
Chunk 18: 500000 rows
Chunk 19: 500000 rows
Chunk 20: 500000 rows
Chunk 21: 500000 rows
Chunk 22: 500000 rows
Chunk 23: 500000 rows
Chunk 24: 500000 rows
Chunk 25: 500000 rows
Chunk 26: 241035 rows


In [10]:
for col in data.columns:
    print(f'Summary statistics for column: {col}')
    print(data[col].describe())
    null_count = data[col].isnull().sum()
    print(f'Null values in column {col}: {null_count}')
    print("\n")

Summary statistics for column: vendor_id
count    3.475226e+06
mean     1.785428e+00
std      4.263282e-01
min      1.000000e+00
25%      2.000000e+00
50%      2.000000e+00
75%      2.000000e+00
max      7.000000e+00
Name: vendor_id, dtype: float64
Null values in column vendor_id: 0


Summary statistics for column: tpep_pickup_datetime
count                       3475226
mean     2025-01-17 11:02:55.910964
min             2024-12-31 20:47:55
25%             2025-01-10 07:59:01
50%             2025-01-17 15:41:33
75%             2025-01-24 19:34:06
max             2025-02-01 00:00:44
Name: tpep_pickup_datetime, dtype: object
Null values in column tpep_pickup_datetime: 0


Summary statistics for column: tpep_dropoff_datetime
count                       3475226
mean     2025-01-17 11:17:56.997901
min             2024-12-18 07:52:40
25%      2025-01-10 08:15:29.500000
50%             2025-01-17 15:59:34
75%             2025-01-24 19:48:31
max             2025-02-01 23:44:11
Name: tpep_drop

In [ ]:
from mage_ai.settings.repo import get_repo_path
from mage_ai.io.config import ConfigFileLoader
from mage_ai.io.postgres import Postgres
from pandas import DataFrame
from os import path

if 'data_exporter' not in globals():
    from mage_ai.data_preparation.decorators import data_exporter

@data_exporter
def export_data_to_postgres(data: DataFrame, **kwargs) -> None:
    schema_name = 'raw'  # Specify the name of the schema to export data to
    table_name = 'ny_taxi_trips'  # Specify the name of the table to export data to
    config_path = path.join(get_repo_path(), 'io_config.yaml')
    config_profile = 'default'

    execution_date = kwargs.get('execution_date')
    if execution_date is None:
        raise ValueError("Failed to get execution_date")

    current_year = int(execution_date.year)
    current_month = int(execution_date.month)

    chunksize = int(kwargs.get("chunksize", 500000))
    rows = data.shape[0]

    with Postgres.with_config(ConfigFileLoader(config_path, config_profile)) as loader:
        print(f"[INFO] Exporting data for {current_year}-{current_month:02d} to PostgreSQL...")

        delete_query = f"""
            DELETE FROM {schema_name}.{table_name}
            WHERE trip_year = {current_year}
              AND trip_month = {current_month};
        """
        try:
            # Borra solo el mes correspondiente, sin leer toda la tabla
            loader.execute(delete_query)
            print(f"[OK] Deleted existing records for {current_year}-{current_month:02d} (if any)")
        except Exception:
            # Si la tabla no existe todavía, simplemente seguimos
            print(f"[ERROR] Failed to delete existing records for {current_year}-{current_month:02d}")
            pass

        for i in range(0, rows, chunksize):
            chunk = data.iloc[i:i + chunksize]

            loader.export(
                chunk,
                schema_name,
                table_name,
                index=False,
                if_exists='append',
            )
            
            print(f"[OK] Exported chunk {i // chunksize + 1} ({chunk.shape[0]} rows) for {current_year}-{current_month:02d}")

        print(f"[OK] Loaded {current_year}-{current_month:02d} ({rows} rows)")
